# Loan Approval — Data Preparation Pipeline

Building a preprocessing and feature engineering pipeline for a loan approval prediction model.  
Using SageMaker Feature Store for feature management and running bias checks before training.

**Dataset:** 50 loan applications with customer demographics, financial info, and approval status.

In [ ]:
!pip install -q boto3 pandas numpy scikit-learn

In [ ]:
import boto3
import pandas as pd
import numpy as np
import time
import json
import os
from sklearn.model_selection import train_test_split

## Setup AWS resources

In [ ]:
region = boto3.session.Session().region_name
sts_client = boto3.client('sts')
caller = sts_client.get_caller_identity()
account_id = caller['Account']
arn = caller['Arn']

# extract role for sagemaker jobs
if ':assumed-role/' in arn:
    role_name = arn.split(':assumed-role/')[1].split('/')[0]
    role = f"arn:aws:iam::{account_id}:role/{role_name}"
else:
    role = arn

bucket = f"sagemaker-{region}-{account_id}"
prefix = "loan-approval-lab"

s3 = boto3.client('s3')
try:
    s3.head_bucket(Bucket=bucket)
except:
    if region == 'us-east-1':
        s3.create_bucket(Bucket=bucket)
    else:
        s3.create_bucket(Bucket=bucket, CreateBucketConfiguration={'LocationConstraint': region})

print(f"Region: {region}")
print(f"Bucket: {bucket}")

## Load and explore the dataset

In [ ]:
df = pd.read_csv('loan_data.csv')

print(f"Shape: {df.shape}")
print(f"\nApproval rate: {df['loan_approved'].mean():.0%}")
print(f"\nMissing values:\n{df.isnull().sum()}")
df.head()

In [ ]:
# quick look at distributions
df.describe()

In [ ]:
# upload raw data to s3
s3.upload_file('loan_data.csv', bucket, f'{prefix}/raw/loan_data.csv')
print(f"Uploaded to s3://{bucket}/{prefix}/raw/loan_data.csv")

## Preprocessing

Steps:
1. Drop ID column
2. Feature engineering — create risk indicators
3. One-hot encode categoricals
4. Train/test split

In [ ]:
# drop customer_id — not a feature
df = df.drop('customer_id', axis=1)

# feature engineering
df['income_per_exp_year'] = df['income'] / (df['employment_years'] + 1)
df['loan_to_income'] = df['loan_amount'] / df['income']
df['risk_score'] = df['late_payments'] * 10 + df['debt_to_income'] * 100

print("New features added: income_per_exp_year, loan_to_income, risk_score")

In [ ]:
# encode categoricals
home_dummies = pd.get_dummies(df['home_ownership'], prefix='home').astype(int)
purpose_dummies = pd.get_dummies(df['purpose'], prefix='purpose').astype(int)

df['gender_male'] = (df['gender'] == 'male').astype(int)
df['gender_female'] = (df['gender'] == 'female').astype(int)

df = pd.concat([df, home_dummies, purpose_dummies], axis=1)
df = df.drop(['home_ownership', 'purpose', 'gender'], axis=1)

# target column first (sagemaker convention)
cols = ['loan_approved'] + [c for c in df.columns if c != 'loan_approved']
df = df[cols]

print(f"Final shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

In [ ]:
# split
train, test = train_test_split(df, test_size=0.2, random_state=42, stratify=df['loan_approved'])

# save locally and to s3
train.to_csv('train.csv', index=False, header=False)
test.to_csv('test.csv', index=False, header=False)
train.to_csv('train_with_headers.csv', index=False)
test.to_csv('test_with_headers.csv', index=False)

s3.upload_file('train.csv', bucket, f'{prefix}/processed/train/train.csv')
s3.upload_file('test.csv', bucket, f'{prefix}/processed/test/test.csv')
s3.upload_file('train_with_headers.csv', bucket, f'{prefix}/processed/train/train_with_headers.csv')

print(f"Train: {train.shape}, Test: {test.shape}")

## Feature Store

Storing processed features in SageMaker Feature Store so they can be reused across experiments and served in real-time during inference.

In [ ]:
train_df = pd.read_csv('train_with_headers.csv')

# add required columns
train_df['record_id'] = range(1, len(train_df) + 1)
train_df['event_time'] = str(pd.Timestamp.now())

# build feature definitions
feature_defs = []
for col in train_df.columns:
    if col == 'record_id':
        feature_defs.append({'FeatureName': col, 'FeatureType': 'Integral'})
    elif col == 'event_time':
        feature_defs.append({'FeatureName': col, 'FeatureType': 'String'})
    elif train_df[col].dtype in ['float64', 'float32']:
        feature_defs.append({'FeatureName': col, 'FeatureType': 'Fractional'})
    else:
        feature_defs.append({'FeatureName': col, 'FeatureType': 'Integral'})

fg_name = f"loan-customers-{int(time.time())}"
sm = boto3.client('sagemaker')

sm.create_feature_group(
    FeatureGroupName=fg_name,
    RecordIdentifierFeatureName='record_id',
    EventTimeFeatureName='event_time',
    FeatureDefinitions=feature_defs,
    OnlineStoreConfig={'EnableOnlineStore': True},
    OfflineStoreConfig={
        'S3StorageConfig': {'S3Uri': f's3://{bucket}/{prefix}/feature-store/'}
    },
    RoleArn=role
)

print(f"Creating feature group: {fg_name}")

# wait
for _ in range(30):
    status = sm.describe_feature_group(FeatureGroupName=fg_name)['FeatureGroupStatus']
    if status != 'Creating':
        break
    time.sleep(10)

print(f"Status: {status}")

In [ ]:
# ingest records
fs_client = boto3.client('sagemaker-featurestore-runtime', region_name=region)

count = 0
for _, row in train_df.iterrows():
    record = [{'FeatureName': col, 'ValueAsString': str(row[col])} 
              for col in train_df.columns if not pd.isna(row[col])]
    fs_client.put_record(FeatureGroupName=fg_name, Record=record)
    count += 1

print(f"Ingested {count} records into feature store")

In [ ]:
# verify — read one record from online store
resp = fs_client.get_record(FeatureGroupName=fg_name, RecordIdentifierValueAsString='1')

print("Sample record from online store:")
for f in resp['Record']:
    print(f"  {f['FeatureName']:25s} = {f['ValueAsString']}")

## Pre-training Bias Analysis

Checking whether there's any gender bias in loan approval labels before we train a model on this data. Using the same metrics that SageMaker Clarify computes internally.

In [ ]:
analysis = train_df.drop(['record_id', 'event_time'], axis=1)

total = len(analysis)
males = len(analysis[analysis['gender_male'] == 1])
females = total - males

male_approved = analysis[analysis['gender_male'] == 1]['loan_approved'].mean()
female_approved = analysis[analysis['gender_male'] == 0]['loan_approved'].mean()

# Class Imbalance
ci = (males - females) / total
print(f"Class Imbalance (CI): {ci:.4f}")
print(f"  Males: {males}, Females: {females}")

# Difference in Proportions of Labels
dpl = male_approved - female_approved
print(f"\nDifference in Proportions (DPL): {dpl:.4f}")
print(f"  Male approval rate:   {male_approved:.0%}")
print(f"  Female approval rate: {female_approved:.0%}")

# Disparate Impact
di = male_approved / female_approved if female_approved > 0 else 0
print(f"\nDisparate Impact (DI): {di:.4f}")
print(f"  Fair range: 0.8 - 1.25 | {'FAIR' if 0.8 <= di <= 1.25 else 'POTENTIAL BIAS'}")

print(f"\n---")
if abs(dpl) > 0.1:
    print("Conclusion: Gender bias detected — consider mitigation before training")
else:
    print("Conclusion: No significant gender bias in approval labels")

## Cleanup

In [ ]:
# uncomment to delete resources
# sm.delete_feature_group(FeatureGroupName=fg_name)
# !aws s3 rm s3://{bucket}/{prefix}/ --recursive
print(f"Feature group to delete: {fg_name}")
print("Remember to stop your JupyterLab space!")